# GPR151 vs GPR88 vs Opioid Receptors: Production-Only Resume

**Optimized for 3-hour GPU window.**

- Skips equilibration (assumes `*_equil.xml` files uploaded as Kaggle input dataset)
- Runs 6 × 20 ns production MD on 2× T4 GPUs
- Estimated time: ~2 hours at GPU speed

**Requirements:** Upload `gpr151_mor_equil.xml`, `gpr151_kor_equil.xml`, `gpr151_dor_equil.xml`, `gpr88_mor_equil.xml`, `gpr88_kor_equil.xml`, `gpr88_dor_equil.xml` as Kaggle input data.


In [1]:
# 0a. pip install (OpenMM via conda in cell 0b, NOT pip)
import sys, subprocess, time, os, shutil
t0 = time.time()
print('Installing for:', sys.executable)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pdbfixer', 'parmed', 'mdanalysis', 'biopython', 'biotite',
    'matplotlib', 'pandas', 'numpy', 'scipy'], check=False)
print(f'pip done in {time.time()-t0:.0f}s')
if shutil.which('obabel') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-q', 'openbabel'], check=True)
    print('obabel installed')
else:
    print('obabel already available')


Installing for: /usr/bin/python3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 55.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.8 MB/s eta 0:00:00
pip done in 26s


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 5 newly installed, 0 to remove and 140 not upgraded.
Need to get 4,148 kB of archives.
After this operation, 19.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-iostreams1.74.0 amd64 1.74.0-14ubuntu3 [245 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libinchi1 amd64 1.03+dfsg-4 [455 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmaeparser1 amd64 1.2.4-1build1 [88.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenbabel7 amd64 3.1.1+dfsg-6ubuntu5 [3,231 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openbabel amd64 3.1.1+dfsg

In [2]:
# 0b. Miniconda + AmberTools + OpenMM (CUDA via conda-forge, driver-aware)
import os, subprocess, time, sys, glob, shutil, re
CONDA_PREFIX = '/opt/miniconda'
AMBER_ENV = f'{CONDA_PREFIX}/envs/ambertools'
OPENMM_ENV = f'{CONDA_PREFIX}/envs/openmm_env'
PY_VER = f'{sys.version_info.major}.{sys.version_info.minor}'
print(f'Python: {sys.executable}')
print(f'Python version: {PY_VER}')

if not os.path.exists(f'{CONDA_PREFIX}/bin/conda'):
    t0 = time.time()
    subprocess.run(['wget', '-q', 'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
                    '-O', '/tmp/miniconda.sh'], check=True)
    subprocess.run(['bash', '/tmp/miniconda.sh', '-b', '-f', '-p', CONDA_PREFIX], check=True)
    os.remove('/tmp/miniconda.sh')
    print(f'Miniconda installed in {time.time()-t0:.0f}s')

conda_bin = f'{CONDA_PREFIX}/bin/conda'

if not os.path.exists(f'{AMBER_ENV}/bin/packmol-memgen'):
    t0 = time.time()
    subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel',
                    'https://repo.anaconda.com/pkgs/main'], check=False)
    subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel',
                    'https://repo.anaconda.com/pkgs/r'], check=False)
    subprocess.run([conda_bin, 'create', '-y', '-q', '-n', 'ambertools',
                    '-c', 'conda-forge', '-c', 'ambermd', 'ambertools'], check=True)
    subprocess.run([conda_bin, 'clean', '-a', '-y'], check=True)
    print(f'AmberTools built in {time.time()-t0:.0f}s')
else:
    print('AmberTools already present')

if not os.path.exists(f'{OPENMM_ENV}/bin/python'):
    t0 = time.time()
    subprocess.run([conda_bin, 'create', '-y', '-q', '-n', 'openmm_env',
                    'python=3.12', '-c', 'conda-forge'], check=True)
    # Check driver version and install matching CUDA toolkit
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print('nvidia-smi output:', r.stdout[:500])
    driver_match = re.search(r'CUDA Version:\s*(\d+\.\d+)', r.stdout)
    if driver_match:
        driver_ver = float(driver_match.group(1))
        print(f'Detected CUDA driver: {driver_ver}')
    else:
        driver_ver = 12.0
        print(f'Could not detect driver version, defaulting to {driver_ver}')
    if driver_ver >= 12.0:
        print('Installing OpenMM with CUDA 12...')
        subprocess.run([conda_bin, 'run', '-n', 'openmm_env',
                        'conda', 'install', '-y', '-q', '-c', 'conda-forge',
                        'openmm', 'cuda-version=12.0'], check=True)
    else:
        print('Installing OpenMM with CUDA 11 (cudatoolkit)...')
        subprocess.run([conda_bin, 'run', '-n', 'openmm_env',
                        'conda', 'install', '-y', '-q', '-c', 'conda-forge',
                        'openmm', 'cudatoolkit'], check=True)
    print(f'OpenMM (CUDA) built in {time.time()-t0:.0f}s')
else:
    print('OpenMM env already present')

PACKMOL_MEMGEN = f'{AMBER_ENV}/bin/packmol-memgen'
TLEAP = f'{AMBER_ENV}/bin/tleap'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
assert os.path.exists(TLEAP), 'tleap not found'
assert os.path.exists(OPENMM_PYTHON), 'openmm_env python not found'
print('AmberTools ready')
print('OpenMM env:', OPENMM_ENV)


Python: /usr/bin/python3
Python version: 3.12
PREFIX=/opt/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/miniconda
Miniconda installed in 17s
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - conda-forge
 - ambermd
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /opt/min

In [ ]:
# 0c. Write the MD engine script (standalone, executed by conda Python)
import os, textwrap
WORK_DIR = '/kaggle/working/gpr151_opioid_pipeline'
os.makedirs(WORK_DIR, exist_ok=True)

MD_ENGINE = textwrap.dedent(r'''#!/usr/bin/env python3
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

os.environ['LD_LIBRARY_PATH'] = (
    '/opt/miniconda/envs/openmm_env/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:'
    + os.environ.get('LD_LIBRARY_PATH', '')
)

import openmm
import openmm.unit as unit
from openmm import Platform, LangevinMiddleIntegrator, CustomExternalForce
from openmm.app import PME, HBonds, AmberPrmtopFile, AmberInpcrdFile, PDBFile, Simulation
from openmm.app import DCDReporter, StateDataReporter, CheckpointReporter
from openmm.app import GBn2, NoCutoff

CONFIG_PATH = sys.argv[1]
with open(CONFIG_PATH) as f:
    cfg = json.load(f)
PHYSICS_MODE = cfg['PHYSICS_MODE']
STEP_SIZE = cfg['STEP_SIZE']
USE_HMASS = cfg['USE_HMASS']
CHECKPOINT_STEPS = cfg['CHECKPOINT_STEPS']
N_STEPS_TOTAL = cfg['N_STEPS_TOTAL']
stage = sys.argv[2]
pair_name = sys.argv[3]
work_dir = sys.argv[4]
gpu_idx = int(sys.argv[5])

def get_platform(gpu_idx):
    platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
    print(f'[{pair_name}] Available platforms: {platforms}')
    if 'CUDA' in platforms:
        print(f'[{pair_name}] Using CUDA platform on GPU {gpu_idx}')
        return Platform.getPlatformByName('CUDA'), {'CudaDeviceIndex': str(gpu_idx), 'CudaPrecision': 'mixed'}
    elif 'OpenCL' in platforms:
        print(f'[{pair_name}] Using OpenCL platform on GPU {gpu_idx}')
        return Platform.getPlatformByName('OpenCL'), {'OpenCLDeviceIndex': str(gpu_idx)}
    else:
        print(f'[{pair_name}] WARNING: No GPU platform available. Using CPU.')
        return Platform.getPlatformByName('CPU'), {}

def run_equil():
    prmtop = AmberPrmtopFile(f'{work_dir}/{pair_name}.prmtop')
    inpcrd = AmberInpcrdFile(f'{work_dir}/{pair_name}.inpcrd')
    if PHYSICS_MODE == 'explicit':
        system = prmtop.createSystem(nonbondedMethod=PME, nonbondedCutoff=1.0*unit.nanometer,
                                     constraints=HBonds, ewaldErrorTolerance=0.0005)
    else:
        system = prmtop.createSystem(implicitSolvent=GBn2, nonbondedMethod=NoCutoff, constraints=HBonds)
    if USE_HMASS:
        for i in range(system.getNumParticles()):
            if system.getParticleMass(i) == 1.008 * unit.amu:
                system.setParticleMass(i, 4.0 * unit.amu)
    force = CustomExternalForce('0.5*k*((x-x0)^2+(y-y0)^2+(z-z0)^2)')
    force.addGlobalParameter('k', 4184.0)
    for p in ['x0','y0','z0']: force.addPerParticleParameter(p)
    for atom in prmtop.topology.atoms():
        if atom.element.symbol != 'H':
            pos = inpcrd.positions[atom.index].value_in_unit(unit.nanometer)
            force.addParticle(atom.index, pos)
    system.addForce(force)
    integrator = LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picosecond, STEP_SIZE*unit.picosecond)
    platform, props = get_platform(gpu_idx)
    simulation = Simulation(prmtop.topology, system, integrator, platform, props)
    simulation.context.setPositions(inpcrd.positions)
    print(f'{pair_name}: Minimizing...')
    simulation.minimizeEnergy(maxIterations=5000)
    print(f'{pair_name}: Heating to 310K...')
    simulation.context.setVelocitiesToTemperature(100*unit.kelvin)
    for temp in [100, 200, 310]:
        integrator.setTemperature(temp*unit.kelvin)
        simulation.step(5000)
    print(f'{pair_name}: Equilibration (10 ns)...')
    simulation.reporters.append(DCDReporter(f'{work_dir}/{pair_name}_equil.dcd', 5000))
    simulation.reporters.append(StateDataReporter(sys.stdout, 5000, step=True,
        potentialEnergy=True, temperature=True, volume=True, speed=True, elapsedTime=True))
    simulation.step(2500000)
    simulation.saveState(f'{work_dir}/{pair_name}_equil.xml')
    positions = simulation.context.getState(getPositions=True).getPositions()
    with open(f'{work_dir}/{pair_name}_equil.pdb', 'w') as f:
        PDBFile.writeFile(prmtop.topology, positions, f)
    print(f'{pair_name}: Equilibration complete')

def run_prod():
    prmtop = AmberPrmtopFile(f'{work_dir}/{pair_name}.prmtop')
    equil_pdb = f'{work_dir}/{pair_name}_equil.pdb'
    if os.path.exists(equil_pdb):
        pdb = PDBFile(equil_pdb)
    else:
        pdb = AmberInpcrdFile(f'{work_dir}/{pair_name}.inpcrd')
    if PHYSICS_MODE == 'explicit':
        system = prmtop.createSystem(nonbondedMethod=PME, nonbondedCutoff=1.0*unit.nanometer,
                                     constraints=HBonds, ewaldErrorTolerance=0.0005)
    else:
        system = prmtop.createSystem(implicitSolvent=GBn2, nonbondedMethod=NoCutoff, constraints=HBonds)
    if USE_HMASS:
        for i in range(system.getNumParticles()):
            if system.getParticleMass(i) == 1.008 * unit.amu:
                system.setParticleMass(i, 4.0 * unit.amu)
    integrator = LangevinMiddleIntegrator(310*unit.kelvin, 1.0/unit.picosecond, STEP_SIZE*unit.picosecond)
    platform, props = get_platform(gpu_idx)
    simulation = Simulation(prmtop.topology, system, integrator, platform, props)
    dcd_path = f'{work_dir}/{pair_name}_prod.dcd'
    chk_file = f'{work_dir}/{pair_name}_prod.chk'
    if os.path.exists(chk_file):
        simulation.loadCheckpoint(chk_file)
        state = simulation.context.getState()
        completed_steps = int(state.getTime().value_in_unit(unit.picosecond) / STEP_SIZE)
        print(f'[{pair_name} GPU{gpu_idx}]: resumed at {completed_steps} steps ({completed_steps*STEP_SIZE/1000:.1f} ns)')
    else:
        simulation.context.setPositions(pdb.positions)
        simulation.context.setVelocitiesToTemperature(310*unit.kelvin)
        completed_steps = 0
        print(f'[{pair_name} GPU{gpu_idx}]: starting fresh')
    simulation.reporters.append(DCDReporter(dcd_path, 25000, append=os.path.exists(dcd_path)))
    log_file = f'{work_dir}/{pair_name}_prod.log'
    simulation.reporters.append(StateDataReporter(log_file, 25000, step=True,
        potentialEnergy=True, temperature=True, speed=True, elapsedTime=True, append=os.path.exists(log_file)))
    simulation.reporters.append(CheckpointReporter(chk_file, CHECKPOINT_STEPS))
    remaining = N_STEPS_TOTAL - completed_steps
    if remaining <= 0:
        print(f'[{pair_name} GPU{gpu_idx}]: already complete')
        return
    print(f'[{pair_name} GPU{gpu_idx}]: running {remaining} steps ({remaining*STEP_SIZE/1000:.1f} ns)')
    try:
        simulation.step(remaining)
        print(f'[{pair_name} GPU{gpu_idx}]: production complete')
    except Exception as e:
        print(f'[{pair_name} GPU{gpu_idx}]: ERROR - {e}')
        import traceback
        traceback.print_exc()

if __name__ == '__main__':
    if stage == 'equil':
        run_equil()
    elif stage == 'prod':
        run_prod()
    else:
        print(f'Unknown stage: {stage}')
''')

with open(f'{WORK_DIR}/md_engine.py', 'w') as f:
    f.write(MD_ENGINE)
print(f'MD engine written to {WORK_DIR}/md_engine.py')


In [ ]:
# 0d. GPU check, working dirs, config, and resume from checkpoint
import subprocess, shutil, os, time, glob, json
if shutil.which('nvidia-smi'):
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print('=== nvidia-smi ===')
    print(r.stdout)
    print('==================')
    r2 = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                       capture_output=True, text=True)
    print('GPU:', r2.stdout.strip())
WORK_DIR = '/kaggle/working/gpr151_opioid_pipeline'
os.makedirs(WORK_DIR, exist_ok=True)
PHYSICS_MODE = 'implicit'
TIME_BUDGET_HOURS = 3
NS_PER_SYSTEM = 20
USE_HMASS = True
STEP_SIZE = 0.004
CHECKPOINT_NS = 5
CHECKPOINT_STEPS = int(CHECKPOINT_NS * 1000 / STEP_SIZE)
N_STEPS_TOTAL = int(NS_PER_SYSTEM * 1000 / STEP_SIZE)
print(f'PHYSICS_MODE: {PHYSICS_MODE}')
print(f'NS_PER_SYSTEM: {NS_PER_SYSTEM} ns')
print(f'STEP_SIZE: {STEP_SIZE} ps ({STEP_SIZE*1000} fs)')
ns_per_hour_per_gpu = 65
total_hours = (6 * NS_PER_SYSTEM) / (ns_per_hour_per_gpu * 2)
print(f'Estimated time: {total_hours:.1f} hours on 2x T4')
RUN_STAGE = {'setup':False, 'structures':False, 'truncation':False, 'complex_pred':False,
             'equil':False, 'production':True, 'analysis':True}
config = {
    'PHYSICS_MODE': PHYSICS_MODE,
    'STEP_SIZE': STEP_SIZE,
    'USE_HMASS': USE_HMASS,
    'CHECKPOINT_STEPS': CHECKPOINT_STEPS,
    'N_STEPS_TOTAL': N_STEPS_TOTAL,
}
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f)
print(f'Config saved to: {CONFIG_PATH}')

# Copy checkpoint files from Kaggle input to working dir
RESUME_INPUT = '/kaggle/input'
copied = 0
for root, dirs, files in os.walk(RESUME_INPUT):
    for f in files:
        if f.endswith('_equil.xml') or f.endswith('.prmtop') or f.endswith('.inpcrd'):
            dest = os.path.join(WORK_DIR, f)
            if not os.path.exists(dest):
                shutil.copy2(os.path.join(root, f), dest)
                copied += 1
                print(f'Copied {f} -> {WORK_DIR}')
if copied > 0:
    print(f'Restored {copied} files from checkpoint input')
else:
    print('WARNING: No checkpoint files found in /kaggle/input. Upload *_equil.xml + .prmtop + .inpcrd as Kaggle input data.')

# List available files
print('\nFiles in working dir:')
for f in os.listdir(WORK_DIR):
    print(f'  {f}')

# Diagnostic: CUDA availability
OPENMM_ENV = '/opt/miniconda/envs/openmm_env'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
diag_script = "import openmm; from openmm import Platform; p=[Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]; print(p)"
diag_env = os.environ.copy()
diag_env['LD_LIBRARY_PATH'] = f'{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:' + diag_env.get('LD_LIBRARY_PATH', '')
r = subprocess.run([OPENMM_PYTHON, '-c', diag_script], capture_output=True, text=True, env=diag_env)
print(f'\nOpenMM platforms (conda): {r.stdout.strip()}')
if r.returncode != 0:
    print(f'ERROR: {r.stderr.strip()}')
elif 'CUDA' in r.stdout:
    print('SUCCESS: CUDA platform is available via conda OpenMM')
else:
    print('WARNING: CUDA not found. Will fall back to OpenCL/CPU.')


In [ ]:
# 0e. Pre-flight benchmark: verify GPU speed before running all simulations
import subprocess, os, time
OPENMM_ENV = '/opt/miniconda/envs/openmm_env'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
WORK_DIR = '/kaggle/working/gpr151_opioid_pipeline'

benchmark_prmtop = None
for f in os.listdir(WORK_DIR):
    if f.endswith('.prmtop'):
        benchmark_prmtop = os.path.join(WORK_DIR, f)
        break

if benchmark_prmtop is None:
    print('WARNING: No prmtop found for benchmark. Skipping benchmark.')
else:
    pair_name = os.path.basename(benchmark_prmtop).replace('.prmtop', '')
    print(f'Benchmarking with {pair_name}...')
    benchmark_script = f'''
import os, sys, time
os.environ['LD_LIBRARY_PATH'] = (
    '{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
import openmm, openmm.unit as unit
from openmm import Platform, LangevinMiddleIntegrator
from openmm.app import AmberPrmtopFile, AmberInpcrdFile, Simulation, HBonds, GBn2, NoCutoff
prmtop = AmberPrmtopFile('{benchmark_prmtop}')
inpcrd = AmberInpcrdFile('{benchmark_prmtop.replace('.prmtop', '.inpcrd')}')
system = prmtop.createSystem(implicitSolvent=GBn2, nonbondedMethod=NoCutoff, constraints=HBonds)
integrator = LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picosecond, 0.004*unit.picosecond)
platform = Platform.getPlatformByName('CUDA')
props = {{'CudaDeviceIndex': '0', 'CudaPrecision': 'mixed'}}
sim = Simulation(prmtop.topology, system, integrator, platform, props)
sim.context.setPositions(inpcrd.positions)
sim.minimizeEnergy(maxIterations=100)
t0 = time.time()
sim.step(5000)  # 20 ps benchmark
elapsed = time.time() - t0
speed_ns_day = (5000 * 0.004 / 1000) / (elapsed / 86400)
print(f'BENCHMARK_SPEED: {{speed_ns_day:.1f}} ns/day')
print(f'BENCHMARK_TIME: {{elapsed:.2f}} s for 20 ps')
'''
    bench_path = f'{WORK_DIR}/benchmark.py'
    with open(bench_path, 'w') as f:
        f.write(benchmark_script)
    my_env = os.environ.copy()
    my_env['LD_LIBRARY_PATH'] = f'{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:' + my_env.get('LD_LIBRARY_PATH', '')
    r = subprocess.run([OPENMM_PYTHON, bench_path], capture_output=True, text=True, env=my_env)
    print(r.stdout)
    if r.returncode != 0:
        print(f'Benchmark error: {r.stderr}')
    else:
        for line in r.stdout.split('\n'):
            if 'BENCHMARK_SPEED' in line:
                speed = float(line.split(':')[1].strip().split()[0])
                print(f'\n*** GPU benchmark speed: {speed:.1f} ns/day ***')
                if speed < 500:
                    print('WARNING: GPU speed is CPU-level. Check CUDA driver/toolkit match.')
                    print('Try: reinstall openmm_env with cuda-version=12.0 instead of cudatoolkit')
                else:
                    print('GPU is performing well. Proceeding with simulations.')


In [ ]:
# 1. Production MD (concurrent on 2 GPUs via conda Python)
import threading, subprocess, os
OPENMM_ENV = '/opt/miniconda/envs/openmm_env'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
MD_ENGINE = f'{WORK_DIR}/md_engine.py'
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
orphan_list = ['gpr151', 'gpr88']
opioid_list = ['mor', 'kor', 'dor']
if RUN_STAGE['production']:
    all_pairs = [f'{o}_{op}' for o in orphan_list for op in opioid_list]
    ready_pairs = [p for p in all_pairs if os.path.exists(f'{WORK_DIR}/{p}.prmtop')
                 and os.path.exists(f'{WORK_DIR}/{p}_equil.xml')]
    if not ready_pairs:
        print('No systems ready for production. Need .prmtop + _equil.xml files.')
    else:
        print(f'Ready for production: {ready_pairs}')
        gpu0_pairs = ready_pairs[:len(ready_pairs)//2]
        gpu1_pairs = ready_pairs[len(ready_pairs)//2:]
        print(f'GPU 0 will run: {gpu0_pairs}')
        print(f'GPU 1 will run: {gpu1_pairs}')
        def gpu_worker(pairs, g_idx):
            for p in pairs:
                print(f'\n--- Starting production for {p} on GPU {g_idx} ---')
                my_env = os.environ.copy()
                my_env['LD_LIBRARY_PATH'] = f'{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:' + my_env.get('LD_LIBRARY_PATH', '')
                r = subprocess.run([OPENMM_PYTHON, MD_ENGINE, CONFIG_PATH, 'prod', p, WORK_DIR, str(g_idx)],
                                   capture_output=False, text=True, env=my_env)
                if r.returncode != 0:
                    print(f'ERROR: production for {p} failed')
        threads = []
        for gpu_idx, gpu_pairs in [(0, gpu0_pairs), (1, gpu1_pairs)]:
            if gpu_pairs:
                t = threading.Thread(target=gpu_worker, args=(gpu_pairs, gpu_idx))
                t.start()
                threads.append(t)
        for t in threads:
            t.join()
        print('All production runs completed')
else:
    print('Skipping production')


In [ ]:
# 2a. Analysis functions
import json, os, MDAnalysis as mda
from MDAnalysis.analysis import distances, align
import numpy as np
def analyze_pair(pair_name, work_dir):
    pdb = f'{work_dir}/{pair_name}_equil.pdb'
    dcd = f'{work_dir}/{pair_name}_prod.dcd'
    if not os.path.exists(dcd):
        print(f'{pair_name}: no trajectory')
        return None
    u = mda.Universe(pdb, dcd)
    split_file = f'{work_dir}/{pair_name}_split.json'
    if os.path.exists(split_file):
        with open(split_file) as jf:
            split = json.load(jf)
        n_A, n_B = split['n_A'], split['n_B']
    else:
        u_complex = mda.Universe(f'{work_dir}/{pair_name}_complex.pdb')
        n_A = len(u_complex.select_atoms('chainid A').residues)
        n_B = len(u_complex.select_atoms('chainid B').residues)
    ca_A = u.select_atoms(f'resid 1-{n_A} and name CA')
    ca_B = u.select_atoms(f'resid {n_A+1}-{n_A+n_B} and name CA')
    all_A = u.select_atoms(f'resid 1-{n_A}')
    all_B = u.select_atoms(f'resid {n_A+1}-{n_A+n_B}')
    com_dists, iface_rmsd, contact_counts, hbond_proxy = [], [], [], []
    u.trajectory[0]
    ref_A = ca_A.positions.copy()
    ref_B = ca_B.positions.copy()
    for ts in u.trajectory:
        com_dists.append(np.linalg.norm(all_A.center_of_mass() - all_B.center_of_mass()))
        R, _ = align.rotation_matrix(ca_A.positions, ref_A)
        aligned_B = np.dot(ca_B.positions - ca_B.positions.mean(axis=0), R) + ref_B.mean(axis=0)
        iface_rmsd.append(np.sqrt(np.mean(np.sum((aligned_B - ref_B)**2, axis=1))))
        dist_mat = distances.distance_array(all_A.positions, all_B.positions)
        contact_counts.append(np.sum(dist_mat < 5.0))
        if ts.frame % 10 == 0:
            donors = all_A.select_atoms('name N* or name O*')
            acceptors = all_B.select_atoms('name N* or name O*')
            d = distances.distance_array(donors.positions, acceptors.positions)
            hbond_proxy.append(np.sum(d < 3.5))
    return {'com': np.array(com_dists), 'rmsd': np.array(iface_rmsd),
            'contacts': np.array(contact_counts), 'hbonds': np.array(hbond_proxy)}


In [ ]:
# 2b. Run analysis for all 6 pairs
orphan_list = ['gpr151', 'gpr88']
opioid_list = ['mor', 'kor', 'dor']
results = {}
for orphan in orphan_list:
    for opioid in opioid_list:
        pair = f'{orphan}_{opioid}'
        data = analyze_pair(pair, WORK_DIR)
        if data is not None:
            results[pair] = data
            print(f"{pair}: COM={data['com'].mean():.1f}±{data['com'].std():.1f} A, "
                  f"RMSD={data['rmsd'].mean():.1f}±{data['rmsd'].std():.1f} A, "
                  f"contacts={data['contacts'].mean():.0f}")
if not results:
    print('WARNING: No trajectories loaded. Run production MD first.')


In [ ]:
# 2c. 6-system comparison dashboard
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('GPR151 vs GPR88 vs Opioid Receptors: Heterodimerization Analysis', fontsize=16)
colors = {
    'gpr151_mor': 'crimson', 'gpr151_kor': 'orangered', 'gpr151_dor': 'darkred',
    'gpr88_mor': 'steelblue', 'gpr88_kor': 'royalblue', 'gpr88_dor': 'navy'
}
for pair, data in results.items():
    c = colors.get(pair, 'gray')
    axes[0,0].plot(data['com'], color=c, label=pair); axes[0,0].set_title('COM Distance (A)'); axes[0,0].legend()
    axes[0,1].plot(data['rmsd'], color=c); axes[0,1].set_title('Interface RMSD (A)'); axes[0,1].set_ylim(0,8)
    axes[0,2].plot(data['contacts'], color=c); axes[0,2].set_title('Contacts (<5A)')
    if len(data['hbonds']) > 0:
        axes[1,0].plot(np.arange(len(data['hbonds']))*10, data['hbonds'], color=c); axes[1,0].set_title('H-bond proxy')
axes[1,1].axis('off'); axes[1,2].axis('off')
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/heterodimerization_dashboard.png', dpi=150)
plt.show()


In [ ]:
# 3. Score each pair and compare GPR151 to GPR88
print('='*70)
print('HETERODIMERIZATION EVIDENCE SUMMARY')
print('='*70)
for opioid in opioid_list:
    p151 = results.get(f'gpr151_{opioid}')
    p88 = results.get(f'gpr88_{opioid}')
    print(f'\n{opioid.upper()}:')
    if p151 and p88:
        print(f'  GPR151: COM={p151["com"].mean():.1f}±{p151["com"].std():.1f} A, contacts={p151["contacts"].mean():.0f}')
        print(f'  GPR88:  COM={p88["com"].mean():.1f}±{p88["com"].std():.1f} A, contacts={p88["contacts"].mean():.0f}')
        if p151['contacts'].mean() > p88['contacts'].mean() + 50:
            print(f'  -> GPR151 shows STRONGER evidence for {opioid} heterodimerization')
        elif p151['contacts'].mean() > p88['contacts'].mean() + 20:
            print(f'  -> GPR151 shows MODERATE evidence for {opioid} heterodimerization')
        else:
            print(f'  -> No significant difference between GPR151 and GPR88')
    elif p151:
        print(f'  GPR151 data available, GPR88 missing')
    elif p88:
        print(f'  GPR88 data available, GPR151 missing')
    else:
        print(f'  No data for this opioid')
